# Resumen de información

## Lectura de la base

Importar librerías necesarias.

In [ ]:
from pathlib import Path
from pandas import read_csv

Crear directorio raíz para manejo de archivos y directorios.

In [ ]:
BASE_DIR = Path().resolve().parent.parent
BASE_DIR

Directorio donde se encuentran las bases de datos.

In [ ]:
DATA_DIR = BASE_DIR / 'data' / 'cirrosis'
DATA_DIR

Cargar la base de datos

In [ ]:
cirrosis = read_csv(DATA_DIR / 'cirrhosis.csv')

## Inspección inicial

Visualización de algunas observaciones

cirrosis.columns = [
    'identificador','dias_en_observacion','status','droga_administrada',
    'edad_dias','genero','presencia_ascitis','presencia_hepatomegalia',
    'lesiones_vasculares','presencia_edemas','bilirubina','colesterol',
    'albumina','cobre','fosfatasa_alcalina','aspartato_aminotransferasa',
    'trigliceridos','plaquetas','protrombina','etapa_cirrosis'
]

In [ ]:
cirrosis.head()

Dimensiones de la información

In [ ]:
cirrosis.shape

## Datos faltantes

In [ ]:
cirrosis.isnull().sum()

In [ ]:
100*cirrosis.isnull().mean()

## Arreglar tipo de datos

De acuerdo a la documentación, la base de datos debe tener los siguientes tipos de variable:

| Nombre de Variable | Tipo        |
|--------------------|-------------|
| ID                 | Entero      |
| N_Days             | Entero      |
| Status             | Categórico  |
| Drug               | Categórico  |
| Age                | Entero      |
| Sex                | Categórico  |
| Ascites            | Categórico  |
| Hepatomegaly       | Categórico  |
| Spiders            | Categórico  |
| Edema              | Categórico  |
| Bilirubin          | Continua    |
| Cholesterol        | Entero      |
| Albumin            | Continua    |
| Copper             | Entero      |
| Alk_Phos           | Continua    |
| SGOT               | Continua    |
| Tryglicerides      | Entero      |
| Platelets          | Entero      |
| Prothrombin        | Continua    |
| Stage              | Categórico  |


Primero veamos los datos que tiene la base

In [ ]:
cirrosis.dtypes

### Variables categóricas

In [ ]:
from pandas import Categorical

In [ ]:
for variable in ['Status','Drug','Sex','Ascites','Hepatomegaly','Spiders','Edema','Stage']:
    print(cirrosis[variable].value_counts())

In [ ]:
categoricas = {
    'Status':['C','CL','D'],
    'Drug':['D-penicillamine','Placebo'],
    'Sex':['M','F'],
    'Ascites':['N','Y'],
    'Hepatomegaly':['N','Y'],
    'Spiders':['N','Y'],
    'Edema':['N','S','Y'],
    'Stage':[1,2,3,4]
}

In [ ]:
for variable in categoricas.keys():
    print(cirrosis[variable].value_counts(normalize=True))
    print('\n')

In [ ]:
dfc = cirrosis.copy()
for variable,categorias in categoricas.items():
    dfc[variable] = Categorical(cirrosis[variable],categories=categorias)

In [ ]:
for variable in categoricas.keys():
    print(dfc[variable].value_counts(normalize=True))
    print('\n')

In [ ]:
dfc.dtypes

### Variables enteras

In [ ]:
from pandas import to_numeric

In [ ]:
enteras = ['ID','N_Days','Age','Cholesterol','Copper','Tryglicerides','Platelets']

In [ ]:
dfc['ID'].astype(bool).dtypes

In [ ]:
dfc['Cholesterol'].astype('int64').dtypes

In [ ]:
dfe = dfc.copy()
for variable in enteras:
    dfe[variable] = to_numeric(dfc[variable],errors='coerce',downcast='integer')

Veamos que las variables Cholesterol, Copper, Tryglicerides y Platelets no se transformaron en enteras.

In [ ]:
dfe.dtypes

In [ ]:
dfe = dfc.copy()
for variable in enteras:
    dfe[variable] = dfc[variable].fillna(-1).astype('int')

In [ ]:
dfe.dtypes

### Variables continuas

In [ ]:
continuas = ['Bilirubin','Albumin','Alk_Phos','SGOT','Prothrombin']

In [ ]:
df = dfe.copy()
for variable in continuas:
    df[variable] = to_numeric(dfc[variable],errors='coerce')

In [ ]:
df.dtypes

## Estadísticas descriptivas

### Variables categóricas

1. Frecuencias absolutas: observaciones para cada categoría.
2. Frecuencias relativas: observaciones de cada caetogía entre el total de observaciones.
3. Moda: categoría con mayor frecuencia.

In [ ]:
df.describe(include='category').T

In [ ]:
for variable in categoricas.keys():
    print(df[variable].value_counts())

In [ ]:
for variable in categoricas.keys():
    print(df[variable].value_counts(normalize=True))

### Variables enteras

1. Min
2. Max
3. Media
4. Mediana
5. Percentiles: valor en el cual ya se acumula cierto porcentaje de observaciones
6. Varianza: 1/(n-1) Suma (observacion - media)^2
7. Desviación estándar
8. Rango intercuartil: percentil.75 - percentil.25 (medida de dispersión entre tus datos)
9. Otras

In [ ]:
df.dtypes

In [ ]:
df.select_dtypes(include='int64').describe().T

In [ ]:
df.select_dtypes(include='int64').describe(percentiles=[.2,.4,.6,.8]).T

In [ ]:
df[['Cholesterol','Copper','Tryglicerides','Platelets']].describe().T

### Variables continuas

In [ ]:
df[continuas].describe().T

# Visualización de datos

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

### Histogramas

In [ ]:
sns.histplot(df['Bilirubin'], bins=5, color='skyblue', kde_kws={'bw_adjust': .5},kde=False)
plt.title("Histograma con curva de densidad")
plt.xlabel("Cantidad de Bilirubina")
plt.ylabel("Frecuencia")
plt.show()

### Diagrama de caja

In [ ]:
sns.boxplot(data=df,x='SGOT', color='lightgreen')
plt.title("Diagrama de caja")
plt.xlabel("Valores")
plt.show()

### Gráfica de dispersión

In [ ]:
sns.scatterplot(data=df,x='Age', y='N_Days', color='purple')
plt.title("Gráfica de Dispersión")
plt.xlabel("Alk_Phos")
plt.ylabel("Prothrombintr")
plt.show()

### Mapa de calor

In [ ]:
correlation_matrix = df[continuas+enteras].corr()

In [ ]:
correlation_matrix

In [ ]:
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm',fmt='.2f',cbar_kws={'label': 'Coeficiente de correlación'}, center=0)
plt.title("Mapa de calor")
plt.show()

### Gráfica de frecuencias

In [ ]:
sns.countplot(x='Status', hue='Sex',data=df, stat='proportion', palette="viridis", hue_order=['F','M'])
plt.title("Supervivencia de pacientes por género")
plt.ylabel("Pacientes")
plt.xlabel("Status")
plt.legend(title="Género")
plt.show()

### Gráfica de barras

In [ ]:
sns.barplot(
    data=df,                # Base de datos a leer
    hue='Status',              # Variable con la que se segmenta
    x='Age',                # Variable a graficar
    legend=True,            # No quiero título la gráfica
    errorbar=('ci', 0.95),  # Graficar el intervalo de confianza
    estimator='min',          # Estimador a calcular
    palette='magma',        # Colores
    hue_order=['CL','C', 'D']
)
plt.title('')
plt.show()

### Gráfica de violín

In [ ]:
sns.violinplot(hue='Stage', y='Age', data=df, palette="pastel", inner="quartile", legend=True)
plt.title("Distribución de edad por Etapa")
plt.xlabel('Etapa')
plt.ylabel('Edad')
plt.show()

### Gráfica de franjas

In [ ]:
sns.stripplot(x='Edema', y='SGOT', data=df, jitter=True, palette='deep')
plt.title('Distribución de SGOT por categoría de edema')
plt.xlabel('Presencia de edema')
plt.ylabel('SGOT')
plt.show()

### Gráfica de enjambre

In [ ]:
sns.swarmplot(x='Spiders', y='Copper', data=df, palette="dark")
plt.title('Swarmplot')
plt.xlabel('Presencia de lesiones vasculares')
plt.ylabel('Nivel de cobre')
plt.show()

### Gráfica de puntos

In [ ]:
sns.pointplot(x='Hepatomegaly', y='Age', hue='Drug', data=df, palette="muted", markers=["o", "s"], dodge=True)
plt.title("Average Fare by Class and Gender")
plt.xlabel('Presencia de hepatomegalia')
plt.ylabel('Edad')
plt.legend(title='Tratamiento',loc='best')
plt.show()

### Catplot

In [ ]:
g = sns.catplot(
    data=df,
    x='Drug',
    y='N_Days',
    hue='Sex',
    col='Status',
    kind='violin',
    split=True,
    palette='coolwarm',
)
g._legend.set_title('Género')
g.fig.suptitle('Supervivencia por tratamiento, status y género',y=1.05)
g.set_axis_labels('Tratamiento','Días en observación')
for ax, title in zip(g.axes.flatten(), [f'Censurados','Trasplantados','Moridos']):
    ax.set_title(title)
plt.show()

### Gráfica de pastel

In [ ]:
labels = categoricas.get('Status')
sizes = df['Status'].value_counts(normalize=True)
colors = sns.color_palette("pastel", n_colors=len(sizes))
explode = [0.1 if ix==0 else 0 for ix,_ in enumerate(sizes)]

plt.figure(figsize=(8, 6))
plt.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.title('Gráfica de pastel')
plt.show()

### Gráfica de dona

In [ ]:
plt.figure(figsize=(8, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140, wedgeprops=dict(width=0.3))
plt.axis('equal')  # Equal aspect ratio ensures that donut is drawn as a circle.
plt.title('Gráfica de dona')
plt.show()

In [ ]:
from lifelines import KaplanMeierFitter
import pandas as pd

In [ ]:
data = {
    'tiempo': df['N_Days'],
    'evento': [1 if x=='D' else 0 for x in df['Status']]
}

In [ ]:
df_KM = pd.DataFrame(data)
df_KM

In [ ]:
kmf = KaplanMeierFitter()

In [ ]:
kmf.fit(durations=df_KM['tiempo'], event_observed=df_KM['evento'])

In [ ]:
kmf.plot_survival_function()
plt.title('Curva de Supervivencia Kaplan-Meier')
plt.xlabel('Tiempo (días)')
plt.ylabel('Función de Supervivencia')
plt.show()

# Transformaciones básicas

## Normalización

$$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

In [ ]:
df = (
    cirrosis
    .select_dtypes(include=['integer','float'])
    .drop(['ID','Stage'],axis=1)
    #.drop(1,axis=0)
)
df.head(5)

In [ ]:
df_normalized = (df - df.min()) / (df.max() - df.min())

In [ ]:
df_normalized.head()

In [ ]:
N_min = min(df[['Age']].values)
N_max = max(df[['Age']].values)
df[['Age']].apply(lambda x: (x - N_min)/(N_max - N_min),axis=0).head(5)

## Estandarización

$$X_{std} = \frac{X - \mu}{\sigma}

In [ ]:
df_standardized = (df - df.mean()) / df.std()

In [ ]:
df_standardized.head()

## Categorización

In [ ]:
import pandas as pd

Bines de longitud fija

In [ ]:
num_bins = 5
df['Bilirubina_cat'] = pd.cut(df['Bilirubin'], bins=num_bins, labels=[f'Bin {i+1}' for i in range(num_bins)])

In [ ]:
df['Bilirubin'].min(),df['Bilirubin'].max()

In [ ]:
(28.0 - 0.3) / 5

In [ ]:
[0.3+(i+1)*5.54 for i in range(5)]

In [ ]:
(
    df[['Bilirubin','Bilirubina_cat']]
    .sort_values(by='Bilirubin')
    [(df['Bilirubin']>5.84) & (df['Bilirubin']<=11.38)]
    .head()
)

In [ ]:
(
    df[['Bilirubin','Bilirubina_cat']]
    .groupby('Bilirubina_cat')
    .count()
)

Bines por cuantiles

In [ ]:
num_bins = 5
df['N_Days_Cuant'] = pd.qcut(df['N_Days'], q=num_bins, labels=[f'Bin {i+1}' for i in range(num_bins)])

In [ ]:
(
    df[['N_Days','N_Days_Cuant']]
    .sort_values(by='N_Days')
    [(df['N_Days']>974.8) & (df['N_Days']<=1434.8)]
)

In [ ]:
(
    df[['N_Days','N_Days_Cuant']]
    .groupby('N_Days_Cuant')
    .count()
)

Bines usando *K-Medias*

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=5, random_state=42)
df['KMeans_Bin'] = kmeans.fit_predict(df['Values'])

Bines basados en árboles

In [ ]:
from sklearn.tree import DecisionTreeRegressor
X = df[['Values']]
y = np.random.randint(1, 5, size=50)  # Random target variable
tree = DecisionTreeRegressor(max_leaf_nodes=5, random_state=42)
tree.fit(X, y)
df['Tree_Bin'] = tree.apply(X).astype(str)

Bines por outliers

In [ ]:
lower_bound = df['SGOT'].quantile(0.05)  # 5th percentile
upper_bound = df['SGOT'].quantile(0.95)  # 95th percentile

def outlier_bin(value):
    if value < lower_bound:
        return 'Outlier - Low'
    elif value > upper_bound:
        return 'Outlier - High'
    else:
        return 'Normal'

df['SGOT_out'] = df['SGOT'].apply(outlier_bin)

In [ ]:
(
    df[['SGOT','SGOT_out']]
    .fillna(0)
    [df['SGOT']>0]
    .sort_values(by='SGOT')
    [df['SGOT_out']=='Normal']
)

# Identificación de outliers

In [ ]:
Q1 = df['SGOT'].quantile(0.25)
Q3 = df['SGOT'].quantile(0.75)
IQR = Q3 - Q1

# Definir límites para identificar outliers
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Identificar outliers
outliers = df[(df['SGOT'] < limite_inferior) | (df['SGOT'] > limite_superior)]

In [ ]:
outliers[['SGOT']]

# Imputación de valores faltantes

In [ ]:
df = read_csv(DATA_DIR / 'cirrhosis.csv')

## Eliminar observaciones

In [ ]:
df_sin_na = df.dropna()
df_sin_na.head(2)

In [ ]:
df_sin_na.isnull().sum()

In [ ]:
df_sin_col_na = df.dropna(axis=1,how='any')
df_sin_col_na.head(2)

In [ ]:
df_sin_col_na.isnull().sum()

In [ ]:
df_sin_col_na = df.dropna(axis=1,how='all')
df_sin_col_na

In [ ]:
df_sin_tre_na = df.dropna(axis=1,thresh=418-135)
df_sin_tre_na

In [ ]:
df_sin_tre_na.isnull().sum()

## Imputar con alguna medida

In [ ]:
df['Copper'] = df['Copper'].fillna(df['Copper'].mean())

In [ ]:
df.isnull().sum()

In [ ]:
df['Tryglicerides'] = df['Tryglicerides'].fillna(df['Tryglicerides'].median())

In [ ]:
df.isnull().sum()

In [ ]:
moda = df['Drug'].mode()[0]
df['Drug'] = df['Drug'].fillna(moda)

In [ ]:
df.isnull().sum()